### University of Virginia
### DS 7200: Distributed Computing
### Lab: Supervised Learning
### Last Updated: August 20, 2023

Sabine Segaloff

---

#### Instructions

This project has two parts:
- Part I: Classification - build and apply a logistic regression model on the Wisconsin Breast Cancer dataset.
- Part II: Regression - build and apply a linear regression model on the California Housing dataset.

**Total Possible Points: 10**

---

#### Part I: Classification (5 POINTS)

Here are the specifications and grading breakdown:

- the target variable is `diagnosis`
- use `f1`, `f2` as predictors (1 PT)
- split data into 60% training set, 40% test set 
- standardize the predictors (1 PT)
- use seed=314 whenever a seed is needed
- fit a Logistic Regression model with an intercept (1 PT)
- compute and show the area under the ROC curve for the test set (2 PTS)

In [30]:
from pyspark.sql import SparkSession

DATA_FILEPATH = 'wisc_breast_cancer_w_fields.csv'

spark = SparkSession \
    .builder \
    .appName("Wisc BRCA") \
    .getOrCreate()

#### Enter code and solution

##### Load data

In [31]:
# load the data into DF

# inferSchema=True has Spark infer types instead of defaulting all to string
# note that it also makes the read not fully lazy (to guess the types Spark has to scan the file right away - this data is small enough that the extra pass over it isn't an issue)
cancer_df = spark.read.csv(DATA_FILEPATH, header=True, inferSchema=True)

# check that f1 and f2 came in as double (numeric) and that diagnosis is string
cancer_df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- f1: double (nullable = true)
 |-- f2: double (nullable = true)
 |-- f3: double (nullable = true)
 |-- f4: double (nullable = true)
 |-- f5: double (nullable = true)
 |-- f6: double (nullable = true)
 |-- f7: double (nullable = true)
 |-- f8: double (nullable = true)
 |-- f9: double (nullable = true)
 |-- f10: double (nullable = true)
 |-- f11: double (nullable = true)
 |-- f12: double (nullable = true)
 |-- f13: double (nullable = true)
 |-- f14: double (nullable = true)
 |-- f15: double (nullable = true)
 |-- f16: double (nullable = true)
 |-- f17: double (nullable = true)
 |-- f18: double (nullable = true)
 |-- f19: double (nullable = true)
 |-- f20: double (nullable = true)
 |-- f21: double (nullable = true)
 |-- f22: double (nullable = true)
 |-- f23: double (nullable = true)
 |-- f24: double (nullable = true)
 |-- f25: double (nullable = true)
 |-- f26: double (nullable = true)
 |-- f27: double (

> Spark's LogisticRegression requires a numeric label (0.0/1.0); it won't accept a string column. diagnosis is stored as "M" (malignant) / "B" (benign), so I need to convert it.

> I'm choosing malignant = 1 on purpose: it's the event we care about detecting, and the evaluator treats 1.0 as the positive class (same idea as event_level = "second" in yardstick).

> Note to self: this is a fixed rule, not something learned from the data, so it's safe to do before the train/test split. No leakage (unlike the scaler, which learns means/SDs and must be fit on train only).

In [32]:
# check what values diagnosis actually takes before converting
# (otherwise(0.0) below would silently label any typo/missing value as benign)
cancer_df.groupBy("diagnosis").count().show()  # .show() is an action so Spark actually runs here

+---------+-----+
|diagnosis|count|
+---------+-----+
|        B|  357|
|        M|  212|
+---------+-----+



In [33]:
from pyspark.sql import functions as F  # F is the normal alias for Spark's column functions

# create numeric label column: malignant (M) = 1.0, benign (B) = 0.0
cancer_df = cancer_df.withColumn(           # withColumn returns a NEW DataFrame (Spark DFs are immutable), so reassign
    "label",                                # "label" is the default labelCol LogisticRegression looks for
    F.when(F.col("diagnosis") == "M", 1.0)  # this works like dplyr's if_else / case_when
     .otherwise(0.0)                        # everything else -> 0.0 (safe to do since groupBy above showed only M and B)
)
# lazy: nothing runs yet (this just adds a step to the plan)

# confirm the mapping across all rows
# each diagnosis should map to exactly one label value
cancer_df.groupBy("diagnosis", "label").count().show()

+---------+-----+-----+
|diagnosis|label|count|
+---------+-----+-----+
|        B|  0.0|  357|
|        M|  1.0|  212|
+---------+-----+-----+



Each diagnosis maps to exactly one label (M to 1.0 and B to 0.0), and the counts match the original 212/357 so my conversion is correct.

##### Assemble predictors

>Spark models only accept a single vector column for the predictors (unlike R's `y ~ f1 + f2` formula), so `VectorAssembler` packs `f1` and `f2` into one `raw_features` column.

>`VectorAssembler` is a transformer with no `.fit()`, so it could safely run before the split. I'm putting it in the pipeline anyway so that all preprocessing lives in one place.

In [34]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["f1", "f2"], # inputCols takes a list of column names (note plural: inputCols)
                            outputCol="raw_features")

cancer_vec_df = assembler.transform(cancer_df) # preview only - the pipeline does the real transform
cancer_vec_df.select("f1", "f2", "raw_features", "label").show(3, truncate=False) # action (spark runs here)

+-----+-----+-------------+-----+
|f1   |f2   |raw_features |label|
+-----+-----+-------------+-----+
|17.99|10.38|[17.99,10.38]|1.0  |
|20.57|17.77|[20.57,17.77]|1.0  |
|19.69|21.25|[19.69,21.25]|1.0  |
+-----+-----+-------------+-----+
only showing top 3 rows


##### Train/test split

I'm splitting `cancer_df` (which has the label but no feature vector yet), because the pipeline will do the assembling and scaling after the split. That's what keeps the scaler from seeing test data. (**Leak prevention**)

`randomSplit([0.6, 0.4], seed=314)` doesn't take an exact 60% sample. Spark flips a weighted coin for each row independently, so the sizes will only be approximately 60/40. It also doesn't stratify (unlike `initial_split(strata = ...)`), so I'll check the class balance in each piece.

In [46]:
# split into train (~60%) and test (~40%) with seed=314 per instructions

# randomSplit takes a list of weights (not necessarily just two; could do [.7, .15, .15] for a train/val/test in one call)
# and returns a LIST of DataFrames (one per weight), so we unpack into two names
train_df, test_df = cancer_df.randomSplit([0.6, 0.4], seed=314)

# cache train_df as it gets reused several times (count, groupBy, pipeline fit) 
# without cache, each action replays the whole plan from the CSV read onward
# see mllib_regression_with_caching.ipynb for more info
# NOTE: .cache() is itself lazy - it only marks the DF; the first action below (.count()) actually fills the cache
# at 569 rows this won't measurably speed anything up - practicing the habit from mllib_regression_with_caching
train_df.cache()
# cache test_df too
test_df.cache()   # also reused: count, groupBy, then transform + evaluate

# check actual sizes - .count() is an action, so Spark runs the plan here
n_train = train_df.count()
n_test = test_df.count()
print(f"train: {n_train}, test: {n_test}, total: {n_train + n_test}")  # total should be 569 (no rows lost or duplicated)
print(f"train fraction: {n_train / (n_train + n_test):.3f}")           # should be close to 0.6, not exact
print(f"test fraction: {n_test / (n_train + n_test):.3f}")           # should be close to 0.4, not exact

26/10/02 15:48:13 WARN CacheManager: Asked to cache already cached data.


train: 354, test: 215, total: 569
train fraction: 0.622
test fraction: 0.378


In [47]:
# randomSplit doesn't stratify, so we should check the M/B balance in each piece
# (overall: 212/569 = ~37% malignant)
train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()

+-----+-----+
|label|count|
+-----+-----+
|  0.0|  216|
|  1.0|  138|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|  141|
|  1.0|   74|
+-----+-----+



In [42]:
138/354 

0.3898305084745763

In [43]:
74/215

0.34418604651162793

The split gave 354 train / 215 test (62.2% train), close to the 60/40 target. The proportion malignant is 39.0% in train and 34.4% in test, close to the overall 37%, so the lack of stratification isn't a problem here.

##### Pipeline stages
The pipeline will run three stages in order, each reading the column the previous one wrote:

| Stage | Type | Learns from data? | Reads → writes |
|---|---|---|---|
| `assembler` | transformer | no | `f1`, `f2` → `raw_features` |
| `scaler` | estimator | yes (mean, SD per feature) | `raw_features` → `features` |
| `lr` | estimator | yes (coefficients, intercept) | `features`, `label` → predictions |

Because `scaler` and `lr` learn from whatever data they're fit on, the whole pipeline gets fit on `train_df` only. The test set only ever gets `.transform()`ed with the training statistics. This is the same idea as `prep(training = ...)` then `bake()` in tidymodels.

In [48]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.classification import LogisticRegression

# scaler is an estimator: .fit() learns each feature's mean and SD (from train only, via the pipeline)
scaler = StandardScaler(inputCol="raw_features",  # reads the assembler's output
                        outputCol="features",     # writes the scaled vector under the name the model reads by default
                        withMean=True,            # subtract the mean (default is False); needed for true standardization
                        withStd=True)             # divide by SD (default True; put it here for clarity)
# withMean=True + withStd=True = (x - mean) / sd
# (my caching notes used withMean=False, which only divides by SD - that's scaling, not standardizing)

# logistic regression is an estimator: .fit() learns the coefficients and intercept
lr = LogisticRegression(featuresCol="features",  # the SCALED vector (not raw_features)
                        labelCol="label",        # numeric 0/1 label I created (M = 1.0)
                        fitIntercept=True)       # True is already the default, but I put it here for clarity
# no regParam/elasticNetParam: we aren't doing regularization, so default (no penalty) applies

# note: defining these objects runs nothing; they're configured tools until the pipeline is fit

In [ ]:
from pyspark.ml import Pipeline

#### Part II: Regression (5 POINTS)

In this project, you will work with the California Home Price dataset to train a regression model and predict median home prices. Here are the specifications and grading breakdown:

- Scale the response variable median_house_value, dividing by 100000 (1 PT)

- Split data into train set (80%), test set (20%) using seed=314 (1 PT)

- Add new predictor: `rooms_per_household`

- In the training set, select all of these features and standardize them: (1 PT)

feats = ["total_bedrooms", 
         "population", 
         "households", 
         "median_income", 
         "rooms_per_household"]

- Fit a linear regression model on the training set with these parameters:

  - maxIter=10
  - regParam=0.3
  - elasticNetParam=0.8  


- Compute the MSE on the test set (2 PTS)

In [35]:
import os
import pandas as pd

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [36]:
DATA_FILEPATH2 = 'cal_housing_data_preproc_w_header.txt'

#### Enter code and solution